# Лабораторная работа №4
## Демонстрация работы автоматизированного workflow
**Тема диплома:** Разработка системы неразрушающего контроля для выявления дефектов металлических бутылок на конвейерной линии

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.webhook_handler import WorkflowClient
from dotenv import load_dotenv

# Загрузка переменных из docker/.env
load_dotenv(os.path.join('..', 'docker', '.env'))
client = WorkflowClient(base_url='http://localhost:5678')

2026-05-05 17:35:59,174 - src.webhook_handler - INFO - WorkflowClient инициализирован: http://localhost:5678/webhook/application


### 1. Проверка доступности n8n

In [21]:
status = client.check_workflow_status()
if status.get('available'):
    print('✅ n8n доступен')
else:
    print(f'❌ n8n недоступен: {status.get("error")}')

✅ n8n доступен


### 2. Базовый workflow: обработка заявки

In [28]:
result = client.send_application(
    message='Не работает вход в систему, ошибка 403',
    contact='user@test.com'
)
if result['success']:
    print('✅ Заявка обработана')
    print('Ответ AI:', result.get('response', {}))
else:
    print(f'❌ Ошибка: {result.get("error")}')

2026-05-05 17:49:24,142 - src.webhook_handler - INFO - Отправка заявки: Не работает вход в систему, ошибка 403...
2026-05-05 17:49:25,095 - src.webhook_handler - INFO - Workflow выполнил обработку


✅ Заявка обработана
Ответ AI: {'category': 'техническая_поддержка', 'original_message': 'Не работает вход в систему, ошибка 403'}


### 3. Специализированный workflow: контроль качества бутылки

In [73]:
import json
import re

def parse_yandexgpt_response(raw_response):
    try:
        text = raw_response['result']['alternatives'][0]['message']['text']
    except (KeyError, IndexError, TypeError):
        return {"verdict": "ошибка", "confidence": 0, "reason": "Не удалось извлечь текст ответа"}
    
    # Ищем JSON внутри текста (модель может обернуть его в ```json ... ```)
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        text = json_match.group(0)
    
    try:
        data = json.loads(text)
        return {
            "verdict": data.get("verdict", "неизвестно"),
            "confidence": data.get("confidence", 0),
            "reason": data.get("reason", "")
        }
    except json.JSONDecodeError:
        return {"verdict": "неизвестно", "confidence": 0, "reason": text}

# --- сам вызов ---
measurements = 'd=65.1mm, h=169.9mm, wall_min=0.22mm, wall_nom=0.30mm, ect_amplitude=18.7mV, anomaly_flag=1'
result = client.send_inspection_data(measurements, 'BTL-DEMO-001')

if result['success']:
    print('✅ Инспекция выполнена')
    parsed = parse_yandexgpt_response(result['response'])
    print('Вердикт:', parsed['verdict'])
    print('Уверенность:', parsed['confidence'])
    print('Причина:', parsed['reason'])
else:
    print(f'❌ Ошибка: {result.get("error")}')

2026-05-05 18:46:19,659 - src.webhook_handler - INFO - Отправка данных бутылки BTL-DEMO-001: d=65.1mm, h=169.9mm, wall_min=0.22mm, wall_nom=0.3...
2026-05-05 18:46:21,363 - src.webhook_handler - INFO - Вердикт: {'result': {'alternatives': [{'message': {'role': 'assistant', 'text': '```\n{\n  "verdict": "Брак_трещина",\n  "confidence": 0.8,\n  "reason": "Высокая амплитуда вихретокового сигнала и наличие аномалий указывают на возможное наличие трещины."\n}\n```'}, 'status': 'ALTERNATIVE_STATUS_FINAL'}], 'usage': {'inputTextTokens': '163', 'completionTokens': '54', 'totalTokens': '217', 'completionTokensDetails': {'reasoningTokens': '0'}}, 'modelVersion': '25.03.2025'}}


✅ Инспекция выполнена
Вердикт: Брак_трещина
Уверенность: 0.8
Причина: Высокая амплитуда вихретокового сигнала и наличие аномалий указывают на возможное наличие трещины.


### 4. Проверка истории выполнения в PostgreSQL

In [74]:
import subprocess
cmd = 'docker exec -it n8n_postgres psql -U n8n -d n8n -c "SELECT id, status, \"startedAt\" FROM execution_entity ORDER BY \"startedAt\" DESC LIMIT 5;"'
subprocess.run(cmd, shell=True)

CompletedProcess(args='docker exec -it n8n_postgres psql -U n8n -d n8n -c "SELECT id, status, "startedAt" FROM execution_entity ORDER BY "startedAt" DESC LIMIT 5;"', returncode=1)

### 5. Выводы
- Workflow автоматизирует обработку заявок и контроль качества.
- AI-классификация выполняется корректно.
- Данные фиксируются в PostgreSQL.